<a href="https://colab.research.google.com/github/VinayaSharada/KateelLearningDemosToStudents/blob/main/TreasuryAnalytics/Colab_Notebooks/04_real_time_anomaly_detection_with_neural_networks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Real-Time Anomaly Detection with Neural Networks

This teaching notebook demonstrates a simplified treasury anomaly workflow using synthetic payments and an unsupervised detector.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(42)


## Step-by-Step Explanation

### What this cell is doing
1. Imports the Python libraries that support data handling, modeling, or visualization in the next steps.
2. Loads or generates the dataset that the rest of the analysis depends on.
3. Key code cues in this cell include `import numpy as np`, which sets the direction for the rest of the cell.

### How to interpret the result
- Use the imported library list to explain which tools are responsible for tables, charts, and model behavior later in the notebook.
- Review the rows and columns carefully because this dataset defines what the model or analysis is allowed to learn from.
- Ask students what business decision would change if this output moved materially up, down, or in an unexpected direction.


In [ ]:
n = 900
timestamps = pd.date_range('2025-01-01', periods=n, freq='H')
amount = rng.lognormal(mean=10.8, sigma=0.55, size=n)
channel = rng.choice(['Supplier Payment', 'Payroll', 'Intercompany', 'FX Settlement'], size=n)
is_fraud = np.zeros(n, dtype=int)
anomaly_idx = rng.choice(np.arange(n), size=18, replace=False)
amount[anomaly_idx] *= rng.choice([0.15, 4.5], size=18)
is_fraud[anomaly_idx] = 1
df = pd.DataFrame({'timestamp': timestamps, 'amount': np.round(amount, 2), 'channel': channel, 'is_fraud': is_fraud})
df.head()


## Step-by-Step Explanation

### What this cell is doing
1. Loads or generates the dataset that the rest of the analysis depends on.
2. Shows an immediate checkpoint so students can verify that the previous transformation worked as expected.
3. Key code cues in this cell include `n = 900`, which sets the direction for the rest of the cell.

### How to interpret the result
- Review the rows and columns carefully because this dataset defines what the model or analysis is allowed to learn from.
- Use this checkpoint to confirm that the structure, sample values, and labels still make business sense.
- Ask students what business decision would change if this output moved materially up, down, or in an unexpected direction.


In [ ]:
scaler = StandardScaler()
amount_scaled = scaler.fit_transform(df[['amount']].values)
detector = IsolationForest(contamination=0.02, random_state=42)
pred = detector.fit_predict(amount_scaled)
df['is_anomaly_detection'] = pred == -1

print(f'Known injected anomalies: {df["is_fraud"].sum()}')
print(f'Detected anomalies: {df["is_anomaly_detection"].sum()}')
print(f'Fraud volume: ₹{df[df["is_fraud"]]["amount"].sum():,.0f}')
df.loc[df['is_anomaly_detection'], ['timestamp', 'amount', 'channel']].head(10)


## Step-by-Step Explanation

### What this cell is doing
1. Trains or evaluates a predictive model using the prepared feature set.
2. Shows an immediate checkpoint so students can verify that the previous transformation worked as expected.
3. Key code cues in this cell include `scaler = StandardScaler()`, which sets the direction for the rest of the cell.

### How to interpret the result
- Treat the reported metrics as decision-support quality indicators, not as proof that the model is production-ready.
- Use this checkpoint to confirm that the structure, sample values, and labels still make business sense.
- Ask students what business decision would change if this output moved materially up, down, or in an unexpected direction.


In [ ]:
plt.figure(figsize=(13, 5))
plt.plot(df['timestamp'], df['amount'], color='#0ea5e9', alpha=0.8, label='Transactions')
subset = df[df['is_anomaly_detection']]
plt.scatter(subset['timestamp'], subset['amount'], color='red', s=40, label='Detected anomalies')
plt.title('Treasury Transaction Anomaly Detection')
plt.xlabel('Timestamp')
plt.ylabel('Amount')
plt.legend()
plt.tight_layout()
plt.show()


## Step-by-Step Explanation

### What this cell is doing
1. Creates a chart so learners can inspect structure, trend, dispersion, or risk visually.
2. Key code cues in this cell include `plt.figure(figsize=(13, 5))`, which sets the direction for the rest of the cell.

### How to interpret the result
- The right interpretation is usually about relative shape, outliers, and direction of movement rather than memorizing exact pixel-level detail.
- Ask students what business decision would change if this output moved materially up, down, or in an unexpected direction.
